# Multi-Armed Bandit (ε-greedy v. UCB)/ Q-Learning
### Name: Brian Pugh
### Topic: Reinforcement Learning — Bandits & Gridworld

## Part 1 — Multi-Armed Bandit (ε-greedy vs UCB)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

n_arms = 4
true_means = rng.normal(loc=[0.3, 0.5, 0.6, 0.4], scale=0.05, size=n_arms)
true_stds  = np.full(n_arms, 0.1)
print('True means:', np.round(true_means, 3))

def pull_arm(a):
    return rng.normal(true_means[a], true_stds[a])

def run_epsilon_greedy(T=1000, eps=0.1):
    Q = np.zeros(n_arms)
    N = np.zeros(n_arms)
    rewards = np.zeros(T)
    for t in range(T):
        if rng.random() < eps:
            a = rng.integers(n_arms)
        else:
            a = int(np.argmax(Q))
        r = pull_arm(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards[t] = r
    return rewards

def run_ucb(T=1000, c=2.0):
    Q = np.zeros(n_arms); N = np.zeros(n_arms)
    rewards = np.zeros(T)
    for a in range(n_arms):
        r = pull_arm(a)
        N[a] += 1
        Q[a] = r
        rewards[a] = r
    for t in range(n_arms, T):
        ucb = Q + c*np.sqrt(np.log(t+1)/(N+1e-9))
        a = int(np.argmax(ucb))
        r = pull_arm(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards[t] = r
    return rewards

T = 1000
rg = run_epsilon_greedy(T, eps=0.1)
ru = run_ucb(T, c=2.0)

plt.figure()
plt.plot(np.cumsum(rg), label='epsilon-greedy (eps=0.1)')
plt.plot(np.cumsum(ru), label='UCB (c=2.0)')
plt.xlabel('Trials'); plt.ylabel('Cumulative reward')
plt.title('Multi-Armed Bandit: Strategy Comparison')
plt.grid(True); plt.legend(); plt.show()

## Part 2 — Q-Learning in Gridworld

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H, W = 5, 5
start = (0, 0)
goal  = (4, 4)
obstacles = {(1,1), (1,2), (2,1), (3,3)}

ACTIONS = [(0,1), (0,-1), (1,0), (-1,0)]
ARROWS  = ['→','←','↓','↑']
nA = len(ACTIONS)

def step(state, action):
    y, x = state
    dy, dx = ACTIONS[action]
    ny, nx = max(0, min(H-1, y+dy)), max(0, min(W-1, x+dx))
    next_state = (ny, nx)
    if next_state in obstacles:
        next_state = state
    reward = 1.0 if next_state == goal else -0.01
    done = (next_state == goal)
    return next_state, reward, done

def to_idx(s): return s[0]*W + s[1]
def from_idx(i): return (i//W, i%W)

gamma = 0.98
alpha = 0.2
eps   = 0.2
episodes = 800

Q = np.zeros((H*W, nA))
episode_rewards = []

rng2 = np.random.default_rng(1)

for ep in range(episodes):
    s = start
    total = 0.0
    for t in range(200):
        si = to_idx(s)
        if rng2.random() < eps:
            a = rng2.integers(nA)
        else:
            a = int(np.argmax(Q[si]))
        ns, r, done = step(s, a)
        nsi = to_idx(ns)
        Q[si, a] += alpha * (r + gamma * np.max(Q[nsi]) - Q[si, a])
        s = ns; total += r
        if done:
            break
    episode_rewards.append(total)

import pandas as pd
sr = pd.Series(episode_rewards).rolling(20).mean()
plt.figure()
plt.plot(episode_rewards, alpha=0.4, label='per-episode reward')
plt.plot(sr, label='rolling mean (w=20)')
plt.xlabel('Episode'); plt.ylabel('Reward'); plt.title('Q-Learning in Gridworld')
plt.grid(True); plt.legend(); plt.show()

policy_grid = np.full((H,W), '·', dtype=object)
for y in range(H):
    for x in range(W):
        s = (y,x)
        if s == goal:
            policy_grid[y,x] = 'G'
        elif s in obstacles:
            policy_grid[y,x] = '■'
        else:
            a = int(np.argmax(Q[to_idx(s)]))
            policy_grid[y,x] = ARROWS[a]

print('Greedy policy (G=goal, ■=obstacle):\n')
for row in policy_grid:
    print(' '.join(row))

## Reflection — Exploration vs. Exploitation in ECE

- **Adaptive Control:** Try new controller gains (explore) while favoring those with better closed-loop reward (exploit). UCB bounds regret.
- **Robotics:** ε-greedy ensures coverage when learning navigation; exploitation follows learned shortest/safest routes.
- **Circuit Tuning:** Selecting bias points/tests resembles pulling arms; exploration reduces uncertainty, exploitation maximizes throughput.

Dimensionality reduction and clustering can precede RL to compress state spaces and boost sample efficiency.